# 02 — Training Dataset (eBay graded images)

Goal: build a dataset of **images + PSA grade labels**.

Key idea:
- eBay is used for *labels and images*, not your primary runtime pricing.


## 1) Collect metadata + images

In [ ]:
!python scripts/collect_training_images.py

In [ ]:
import pandas as pd
from pathlib import Path
import yaml
cfg = yaml.safe_load(open('config/config.yaml'))
meta = pd.read_csv(Path(cfg['raw_data'])/'training_images'/'metadata.csv')
meta.head()

## 2) Dataset checks
You want:
- balanced grades (or class weighting)
- variety across cards and sellers
- filtered out 'lot/bundle/proxy'


In [ ]:
meta['psa_grade'].value_counts().sort_index()

## Quick setup (optional)

If you hosted a small sample training set on Drive, you can download it:

```bash
python scripts/download_assets.py --asset sample_images
```

Otherwise, generate your training set from eBay:

```bash
python scripts/collect_training_images.py
```

In [ ]:
from pathlib import Path
import pandas as pd

meta_path = Path('data/raw/training_images/metadata.csv')
img_dir = Path('data/raw/training_images/images')
meta_path.exists(), img_dir.exists()

In [ ]:
import pandas as pd
from pathlib import Path

if not meta_path.exists():
    raise FileNotFoundError('Training metadata missing. Run: python scripts/collect_training_images.py')

df = pd.read_csv(meta_path)
df.head()

## Grade distribution

A grader model is only as good as the label distribution.

If PSA 10 dominates, the model will learn to “predict 10” too often.
This plot tells you whether you need balancing (class weights / sampling).

In [ ]:
import matplotlib.pyplot as plt

counts = df['psa_grade'].value_counts().sort_index()
plt.figure(figsize=(6,3))
plt.bar(counts.index.astype(str), counts.values)
plt.xlabel('PSA grade (label)')
plt.ylabel('Count')
plt.title('Training Label Distribution')
plt.tight_layout()
plt.show()

counts

## Quick visual inspection

Show a small grid of images for one grade. This catches:
- wrong crops (slab label in view)
- broken downloads
- mislabeled listings

If the images include the PSA label, you should later crop/warp to card-only to prevent the model from cheating.

In [ ]:
import random
from PIL import Image
import matplotlib.pyplot as plt

# pick a grade that exists
grade = int(counts.index[0]) if len(counts) else 9
subset = df[df['psa_grade'] == grade].dropna(subset=['image_path'])
paths = subset['image_path'].tolist()
random.shuffle(paths)
paths = paths[:9]

plt.figure(figsize=(6,6))
for i, p in enumerate(paths):
    try:
        img = Image.open(p).convert('RGB')
    except Exception:
        continue
    ax = plt.subplot(3,3,i+1)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'PSA {grade}')
plt.tight_layout()
plt.show()

## Data leakage checklist

Before training, you should implement:
- **dedupe**: perceptual hash (pHash)
- **group split**: split by `item_id` or pHash cluster

This prevents the same listing photo from appearing in train and test.

Portfolio tip: include a small table in your README showing how many duplicates you removed.